# 06 — Probabilistic validation and promotion

This is an output-free release template for the frozen probabilistic typing lane. It rebuilds no scientific algorithm in notebook cells. Library contracts own candidate validation, two-stage survey-bootstrap intervals, multiplicity, policy gates, exact calibration source replay, and the signed promotion receipt.

The enforced order is: build and freeze the complete validation report (broad, all-cell specific, and every probabilistic subtype), pass the exact preregistered policy, explicitly change `OPEN_LOCKED_TEST = False` only under the study release protocol, build the locked report once, and create the signed promotion manifest. A failure is reported; it is never tuned away on validation or locked donors. All action flags default to false, all paths are placeholders, and committed notebook outputs must remain empty.


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from phenocycler.artifacts import RunManifest
from phenocycler.candidate_evaluation import load_candidate_evaluation_manifest
from phenocycler.config import load_config
from phenocycler.evaluation_reporting import (
    EvaluationTargetReview,
    build_split_evaluation_report,
)
from phenocycler.hierarchical_typing import TypingRegistry
from phenocycler.marker_registry import load_registry
from phenocycler.refinement_contracts import ReviewLabelProvenance
from phenocycler.threshold_selection import ThresholdSelectionReplayInputs
from phenocycler.typing_model_bundle import (
    PromotionGatePolicy,
    SplitEvaluationReport,
    TypingModelPromotionManifest,
    load_typing_model_bundle,
)

CONFIG_PATH = None
PRELIMINARY_BUNDLE_PATH = None
FROZEN_BUNDLE_PATH = None
SOURCE_RUN_MANIFEST_PATH = None
CALIBRATION_CANDIDATE_MANIFEST_PATH = None
VALIDATION_CANDIDATE_MANIFEST_PATH = None
LOCKED_TEST_CANDIDATE_MANIFEST_PATH = None
PROMOTION_POLICY_PATH = None
VALIDATION_REPORT_PATH = None
LOCKED_TEST_REPORT_PATH = None
PROMOTION_MANIFEST_PATH = None
CHALLENGE_DIAGNOSTIC_PATH = None

# Each value is {'sample': Path, 'reviewer': Path, 'ledger': Path, 'provenance': Path}.
# Calibration keys use ('broad', '') and ('subtype', '<parent>').
# Held-out maps additionally require ('specific', None).
CALIBRATION_REVIEW_ARTIFACTS = {}
VALIDATION_REVIEW_ARTIFACTS = {}
LOCKED_TEST_REVIEW_ARTIFACTS = {}

LOAD_CONTEXT = False
BUILD_VALIDATION_REPORT = False
LOAD_EXISTING_VALIDATION_REPORT = False
PLOT_REPORTS = False
CREATE_PROMOTION = False
PROMOTION_VERSION = 'pancreas-typing-release-v1'
PROMOTION_ISSUER = None
REVIEW_KEY_ENV = 'PHENOCYCLER_REVIEW_BLINDING_KEY_FILE'
RELEASE_KEY_ENV = 'PHENOCYCLER_RELEASE_SIGNING_KEY_FILE'


## Protected input helpers

Review and signing keys live in protected files outside the repository. Only their preregistered SHA-256 commitments belong in the donor split. The helper returns bytes without logging the path contents or secret. Reviewer-frame context attributes are restored from their strict provenance sidecars after tabular serialization; release signing independently reconstructs those frames from the sole content-validated ingest snapshot. When an immutable container or solved environment is available, set `PHENOCYCLER_RUNTIME_IMAGE_DIGEST` to its lowercase SHA-256 digest before promotion and production loading; it augments the automatically hashed installed numerical runtime.


In [ ]:
def _read_table(path):
    path = Path(path).expanduser()
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'unsupported review table: {path.suffix}')


def _read_protected_key(environment_name):
    configured = os.environ.get(environment_name, '')
    if not configured:
        raise RuntimeError(f'{environment_name} must name a protected key file')
    key_path = Path(configured).expanduser()
    if not key_path.is_file():
        raise FileNotFoundError(f'protected key file is unavailable for {environment_name}')
    key = key_path.read_bytes()
    if len(key) < 16:
        raise ValueError(f'{environment_name} key must contain at least 16 bytes')
    return key


def _load_target_reviews(artifact_map):
    reviews = []
    for target, paths in sorted(artifact_map.items(), key=lambda item: str(item[0])):
        provenance = ReviewLabelProvenance.read_json(paths['provenance'])
        sample_frame = _read_table(paths['sample'])
        reviewer_frame = _read_table(paths['reviewer'])
        reviewer_frame.attrs['review_context_artifact_content_id'] = (
            provenance.review_context_artifact_content_id
        )
        reviews.append(EvaluationTargetReview(
            sample_frame=sample_frame,
            reviewer_frame=reviewer_frame,
            review_labels=_read_table(paths['ledger']),
            provenance=provenance,
        ))
    return tuple(reviews)


def _load_calibration_frames(artifact_map, field):
    frames = {}
    for target, paths in artifact_map.items():
        frame = _read_table(paths[field])
        if field == 'reviewer':
            provenance = ReviewLabelProvenance.read_json(paths['provenance'])
            frame.attrs['review_context_artifact_content_id'] = (
                provenance.review_context_artifact_content_id
            )
        frames[target] = frame
    return frames


def _require_paths(*values):
    if any(value is None for value in values):
        raise ValueError('configure every required immutable artifact path first')


## Load immutable release context

Set `LOAD_CONTEXT=True` only after filling the paths. This validates the preliminary and frozen model bundles against the current marker and typing registries, reads the exact source run, and reads the preregistered promotion policy. It does not open either held-out label set.


In [ ]:
cfg = marker_registry = typing_registry = None
preliminary_bundle = frozen_bundle = source_run_manifest = promotion_policy = None
if LOAD_CONTEXT:
    _require_paths(
        CONFIG_PATH, PRELIMINARY_BUNDLE_PATH, FROZEN_BUNDLE_PATH,
        SOURCE_RUN_MANIFEST_PATH, PROMOTION_POLICY_PATH,
    )
    cfg = load_config(CONFIG_PATH)
    marker_registry = load_registry(cfg.marker_registry)
    typing_registry = TypingRegistry.from_marker_registry(
        marker_registry, typing_rules=cfg.typing_rules,
    )
    preliminary_bundle = load_typing_model_bundle(
        PRELIMINARY_BUNDLE_PATH,
        marker_registry=marker_registry, typing_registry=typing_registry,
    )
    frozen_bundle = load_typing_model_bundle(
        FROZEN_BUNDLE_PATH,
        marker_registry=marker_registry, typing_registry=typing_registry,
    )
    source_run_manifest = RunManifest.read_json(SOURCE_RUN_MANIFEST_PATH)
    promotion_policy = PromotionGatePolicy.read_json(PROMOTION_POLICY_PATH)
    if preliminary_bundle.threshold_selection_provenance is not None:
        raise ValueError('threshold source replay requires the original unfrozen bundle')
    if frozen_bundle.threshold_selection_provenance is None:
        raise ValueError('held-out evaluation requires the threshold-frozen bundle')
    if preliminary_bundle.model_fit_content_id != frozen_bundle.model_fit_content_id:
        raise ValueError('preliminary and frozen bundles do not share one scientific fit')


## Freeze validation before opening the locked test

The report builder requires exactly one broad review, one all-cell specific review, and one review for every fitted probabilistic subtype. It derives every estimate and interval from embedded reviewed rows. `validate_candidate_source_replay()` additionally replays candidate inference, deterministic HMAC sampling, and the exact reviewer projection from the sole ingest snapshot. `validate_for_promotion()` is the supported pre-lock validation gate; if it raises, stop and report the failure.


In [ ]:
validation_report = None
VALIDATION_GATE_PASSED = False
if BUILD_VALIDATION_REPORT:
    if not LOAD_CONTEXT:
        raise RuntimeError('load the immutable release context first')
    _require_paths(VALIDATION_CANDIDATE_MANIFEST_PATH, VALIDATION_REPORT_PATH)
    review_blinding_key = _read_protected_key(REVIEW_KEY_ENV)
    try:
        validation_candidate = load_candidate_evaluation_manifest(
            VALIDATION_CANDIDATE_MANIFEST_PATH, model_bundle=frozen_bundle,
            marker_registry=marker_registry, typing_registry=typing_registry,
            source_run_manifest=source_run_manifest, allow_locked_test=False,
        )
        validation_report = build_split_evaluation_report(
            candidate_evaluation=validation_candidate, model_bundle=frozen_bundle,
            marker_registry=marker_registry, typing_registry=typing_registry,
            source_run_manifest=source_run_manifest,
            target_reviews=_load_target_reviews(VALIDATION_REVIEW_ARTIFACTS),
            promotion_policy=promotion_policy,
            review_blinding_key=review_blinding_key, allow_locked_test=False,
        )
        if validation_report.split_name != 'validation':
            raise ValueError('validation artifact resolved to the wrong split')
        validation_report.validate_candidate_source_replay(
            model_bundle=frozen_bundle, marker_registry=marker_registry,
            typing_registry=typing_registry,
            review_blinding_key=review_blinding_key,
        )
    finally:
        del review_blinding_key
    validation_report.write_json(VALIDATION_REPORT_PATH)
    validation_report.validate_for_promotion(
        model_bundle=frozen_bundle, policy=promotion_policy,
    )
    VALIDATION_GATE_PASSED = True
elif LOAD_EXISTING_VALIDATION_REPORT:
    if not LOAD_CONTEXT:
        raise RuntimeError('load the immutable release context first')
    _require_paths(VALIDATION_REPORT_PATH)
    validation_report = SplitEvaluationReport.read_json(VALIDATION_REPORT_PATH)
    if validation_report.split_name != 'validation':
        raise ValueError('saved validation report has the wrong split')
    review_blinding_key = _read_protected_key(REVIEW_KEY_ENV)
    try:
        validation_report.validate_candidate_source_replay(
            model_bundle=frozen_bundle, marker_registry=marker_registry,
            typing_registry=typing_registry,
            review_blinding_key=review_blinding_key,
        )
    finally:
        del review_blinding_key
    validation_report.validate_for_promotion(
        model_bundle=frozen_bundle, policy=promotion_policy,
    )
    VALIDATION_GATE_PASSED = True


## Plotted release decisions

These plots consume only replayed report fields. Points are estimates; whiskers are the report's simultaneous intervals; dashed lines are the exact preregistered policy gates. Coverage and rescue coverage use lower-bound decisions. Selective errors and one-vs-rest false-positive/false-negative rates use upper-bound decisions. An unresolved or under-resolved interval remains fail-closed rather than disappearing from the plot.


In [ ]:
def _target_name(target):
    return target.level if target.parent is None else f'{target.level}:{target.parent}'


def _interval_rows(report):
    rows = []
    for target in report.targets:
        for metric in ('coverage', 'rescue_coverage', 'selective_error', 'rescue_selective_error'):
            interval = getattr(target, metric)
            rows.append({
                'target': _target_name(target), 'metric': metric,
                'estimate': interval.estimate, 'lower': interval.lower,
                'upper': interval.upper,
            })
    return pd.DataFrame(rows)


def _class_error_rows(report):
    rows = []
    for target in report.targets:
        for result in target.class_errors:
            for metric, interval in (
                ('false_positive_rate', result.false_positive_rate),
                ('false_negative_rate', result.false_negative_rate),
            ):
                rows.append({
                    'target': _target_name(target), 'class': result.label,
                    'metric': metric, 'estimate': interval.estimate,
                    'lower': interval.lower, 'upper': interval.upper,
                })
    return pd.DataFrame(rows)


def plot_release_intervals(report, policy, title_prefix):
    target_metrics = _interval_rows(report)
    class_errors = _class_error_rows(report)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    for axis, metrics, title in (
        (axes[0], ('coverage', 'rescue_coverage'), 'coverage: lower bound must clear gate'),
        (axes[1], ('selective_error', 'rescue_selective_error'), 'error: upper bound must clear gate'),
    ):
        subset = target_metrics[target_metrics['metric'].isin(metrics)].reset_index(drop=True)
        x = np.arange(len(subset))
        axis.errorbar(
            x, subset['estimate'],
            yerr=np.vstack((subset['estimate'] - subset['lower'], subset['upper'] - subset['estimate'])),
            fmt='o', capsize=3, color='#264653',
        )
        axis.set_xticks(x, (subset['target'] + '\n' + subset['metric']).tolist(), rotation=45, ha='right')
        axis.set_ylim(0, 1)
        axis.set_title(f'{title_prefix} — {title}')
        axis.grid(axis='y', alpha=0.2)
    axes[0].axhline(policy.minimum_coverage_lower, ls='--', color='#2a9d8f', label='coverage gate')
    axes[0].axhline(policy.minimum_rescue_coverage_lower, ls=':', color='#e9c46a', label='rescue gate')
    axes[1].axhline(policy.maximum_selective_error_upper, ls='--', color='#e76f51', label='error gate')
    axes[1].axhline(policy.maximum_rescue_selective_error_upper, ls=':', color='#f4a261', label='rescue-error gate')
    for axis in axes:
        axis.legend()
    plt.show()

    labels = class_errors['target'] + ':' + class_errors['class'] + '\n' + class_errors['metric']
    x = np.arange(len(class_errors))
    fig, ax = plt.subplots(figsize=(max(12, 0.35 * len(class_errors)), 5), constrained_layout=True)
    ax.errorbar(
        x, class_errors['estimate'],
        yerr=np.vstack((class_errors['estimate'] - class_errors['lower'], class_errors['upper'] - class_errors['estimate'])),
        fmt='o', capsize=3, color='#457b9d',
    )
    ax.axhline(policy.maximum_class_false_positive_rate_upper, ls='--', color='#e76f51', label='FP upper gate')
    ax.axhline(policy.maximum_class_false_negative_rate_upper, ls=':', color='#9b5de5', label='FN upper gate')
    ax.set_xticks(x, labels.tolist(), rotation=75, ha='right')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Sampling-weighted rate and simultaneous interval')
    ax.set_title(f'{title_prefix} — per-class false positives and false negatives')
    ax.legend()
    ax.grid(axis='y', alpha=0.2)
    plt.show()


if PLOT_REPORTS and validation_report is not None:
    plot_release_intervals(validation_report, promotion_policy, 'validation')


## Challenge-sample diagnostic (never a headline gate)

An enriched challenge sample is useful for discovering rare segmentation, spillover, unavailable-marker, ontology, broad-routing, and subtype failure mechanisms. Because its inclusion probabilities do not estimate the target population, its counts and apparent error rates are diagnostic only. It is not a software-enforced headline metric gate. Challenge data are not passed to `build_split_evaluation_report()`, `validate_for_promotion()`, or `TypingModelPromotionManifest.create()`. If a diagnostic is supplied, this notebook requires exact validation-candidate lineage and an explicit categorical disposition. A catastrophic finding blocks this candidate and requires investigation at the earliest responsible stage plus a new development cycle—not a cell-level override.


In [ ]:
CHALLENGE_DIAGNOSTIC_ONLY = True
challenge_diagnostic = None
CHALLENGE_DISPOSITION_READY = CHALLENGE_DIAGNOSTIC_PATH is None
if CHALLENGE_DIAGNOSTIC_PATH is not None:
    if validation_report is None:
        raise RuntimeError('load or build the exact validation report before its challenge diagnostic')
    challenge_diagnostic = _read_table(CHALLENGE_DIAGNOSTIC_PATH)
    required = {
        'root_cause', 'evidence_sufficient', 'catastrophic', 'challenge_disposition',
        'candidate_assignment_run_id', 'candidate_evaluation_split_name',
        'typing_model_bundle_content_id', 'candidate_evaluation_manifest_content_id',
        'candidate_assignment_output_content_id',
    }
    if not required.issubset(challenge_diagnostic.columns):
        raise KeyError(f'challenge diagnostic is missing {sorted(required - set(challenge_diagnostic.columns))}')
    expected_lineage = {
        'candidate_assignment_run_id': validation_report.assignment_run_id,
        'candidate_evaluation_split_name': 'validation',
        'typing_model_bundle_content_id': validation_report.model_bundle_content_id,
        'candidate_evaluation_manifest_content_id': validation_report.candidate_evaluation_manifest_content_id,
        'candidate_assignment_output_content_id': validation_report.assignment_artifact_content_id,
    }
    for column, expected in expected_lineage.items():
        observed = set(challenge_diagnostic[column].astype(str).str.strip())
        if observed != {expected}:
            raise ValueError(f'challenge diagnostic {column} differs from validation lineage')
    for column in ('evidence_sufficient', 'catastrophic'):
        if challenge_diagnostic[column].isna().any() or not challenge_diagnostic[column].isin([True, False, 0, 1]).all():
            raise ValueError(f'challenge diagnostic {column} must contain explicit booleans')
        challenge_diagnostic[column] = challenge_diagnostic[column].astype(bool)
    if challenge_diagnostic['root_cause'].astype(str).str.strip().eq('').any():
        raise ValueError('challenge diagnostic root_cause must be nonempty')
    disposition = challenge_diagnostic['challenge_disposition'].astype(str).str.strip()
    allowed_dispositions = {'no_catastrophic_finding', 'new_development_cycle_required'}
    if not set(disposition).issubset(allowed_dispositions):
        raise ValueError('challenge diagnostic has an unknown disposition')
    catastrophic = challenge_diagnostic['catastrophic']
    if (catastrophic & disposition.ne('new_development_cycle_required')).any():
        raise ValueError('every catastrophic challenge finding requires a new development cycle')
    if ((~catastrophic) & disposition.ne('no_catastrophic_finding')).any():
        raise ValueError('noncatastrophic challenge rows have an inconsistent disposition')
    CHALLENGE_DISPOSITION_READY = not bool(catastrophic.any())
    if not CHALLENGE_DISPOSITION_READY:
        print('STOP: catastrophic challenge findings require a new candidate and future locked test.')
    if PLOT_REPORTS:
        challenge_counts = (
            challenge_diagnostic.groupby(['root_cause', 'evidence_sufficient'], observed=True)
            .size().rename('reviewed_cells').reset_index()
        )
        pivot = challenge_counts.pivot(index='root_cause', columns='evidence_sufficient', values='reviewed_cells').fillna(0)
        pivot.plot.barh(stacked=True, figsize=(10, max(4, 0.35 * len(pivot))), title='Challenge audit root causes — diagnostic only')
        plt.xlabel('Enriched reviewed cells; not a population rate')
        plt.tight_layout()
        plt.show()


## Explicit one-time locked-test boundary

Do not edit the flag until the validation report has passed, been written immutably, and been archived under the study protocol. The flag is an explicit acknowledgement, not a cryptographic one-use lock. Access controls and the decision ledger must prevent candidate regeneration after results are inspected.


In [ ]:
OPEN_LOCKED_TEST = False
BUILD_LOCKED_TEST_REPORT = False
LOAD_EXISTING_LOCKED_TEST_REPORT = False
locked_test_report = None
if (BUILD_LOCKED_TEST_REPORT or LOAD_EXISTING_LOCKED_TEST_REPORT) and not OPEN_LOCKED_TEST:
    raise PermissionError('locked test remains closed; preserve the frozen validation decision')
if OPEN_LOCKED_TEST:
    if not VALIDATION_GATE_PASSED or validation_report is None:
        raise RuntimeError('a frozen passing validation report is required before locked test')
    _require_paths(VALIDATION_REPORT_PATH)
    archived_validation_report = SplitEvaluationReport.read_json(VALIDATION_REPORT_PATH)
    if archived_validation_report.content_id != validation_report.content_id:
        raise ValueError('in-memory validation report differs from its immutable archive')
    review_blinding_key = _read_protected_key(REVIEW_KEY_ENV)
    try:
        archived_validation_report.validate_candidate_source_replay(
            model_bundle=frozen_bundle, marker_registry=marker_registry,
            typing_registry=typing_registry,
            review_blinding_key=review_blinding_key,
        )
    finally:
        del review_blinding_key
    archived_validation_report.validate_for_promotion(
        model_bundle=frozen_bundle, policy=promotion_policy,
    )
    validation_report = archived_validation_report
    if BUILD_LOCKED_TEST_REPORT:
        if not LOAD_CONTEXT:
            raise RuntimeError('load the immutable release context first')
        _require_paths(LOCKED_TEST_CANDIDATE_MANIFEST_PATH, LOCKED_TEST_REPORT_PATH)
        review_blinding_key = _read_protected_key(REVIEW_KEY_ENV)
        try:
            locked_test_candidate = load_candidate_evaluation_manifest(
                LOCKED_TEST_CANDIDATE_MANIFEST_PATH, model_bundle=frozen_bundle,
                marker_registry=marker_registry, typing_registry=typing_registry,
                source_run_manifest=source_run_manifest, allow_locked_test=True,
            )
            locked_test_report = build_split_evaluation_report(
                candidate_evaluation=locked_test_candidate, model_bundle=frozen_bundle,
                marker_registry=marker_registry, typing_registry=typing_registry,
                source_run_manifest=source_run_manifest,
                target_reviews=_load_target_reviews(LOCKED_TEST_REVIEW_ARTIFACTS),
                promotion_policy=promotion_policy,
                review_blinding_key=review_blinding_key, allow_locked_test=True,
            )
            if locked_test_report.split_name != 'locked_test':
                raise ValueError('locked-test artifact resolved to the wrong split')
            locked_test_report.validate_candidate_source_replay(
                model_bundle=frozen_bundle, marker_registry=marker_registry,
                typing_registry=typing_registry,
                review_blinding_key=review_blinding_key,
            )
        finally:
            del review_blinding_key
        locked_test_report.write_json(LOCKED_TEST_REPORT_PATH)
        locked_test_report.validate_for_promotion(
            model_bundle=frozen_bundle, policy=promotion_policy,
        )
    elif LOAD_EXISTING_LOCKED_TEST_REPORT:
        _require_paths(LOCKED_TEST_REPORT_PATH)
        locked_test_report = SplitEvaluationReport.read_json(LOCKED_TEST_REPORT_PATH)
        if locked_test_report.split_name != 'locked_test':
            raise ValueError('saved locked-test report has the wrong split')
        review_blinding_key = _read_protected_key(REVIEW_KEY_ENV)
        try:
            locked_test_report.validate_candidate_source_replay(
                model_bundle=frozen_bundle, marker_registry=marker_registry,
                typing_registry=typing_registry,
                review_blinding_key=review_blinding_key,
            )
        finally:
            del review_blinding_key
        locked_test_report.validate_for_promotion(
            model_bundle=frozen_bundle, policy=promotion_policy,
        )

if PLOT_REPORTS and OPEN_LOCKED_TEST and locked_test_report is not None:
    plot_release_intervals(locked_test_report, promotion_policy, 'locked test')


## Exact calibration replay and signed promotion

Promotion replays the original preliminary calibration candidate from its immutable source, reconstructs every calibration reviewer frame from the sole ingest artifact, revalidates deterministic HMAC samples and ledgers, and requires the rebuilt threshold provenance to equal the provenance attached to the frozen bundle. The same call source-replays both held-out reports. Its HMAC receipt binds the calibration candidate/output/source identities, both reports, registries, policy, scientific fit, complete package hash, and installed-runtime hash. The release signing key is loaded only inside this guarded cell and deleted immediately; it is never displayed, printed, serialized, or placed in notebook metadata.


In [ ]:
promotion_manifest = None
if CREATE_PROMOTION:
    if not OPEN_LOCKED_TEST:
        raise PermissionError('promotion requires the explicitly opened locked-test workflow')
    if not VALIDATION_GATE_PASSED or validation_report is None or locked_test_report is None:
        raise RuntimeError('promotion requires frozen passing validation and locked-test reports')
    if not CHALLENGE_DISPOSITION_READY:
        raise RuntimeError('catastrophic challenge findings require a new development cycle')
    if not LOAD_CONTEXT:
        raise RuntimeError('load the immutable release context first')
    _require_paths(CALIBRATION_CANDIDATE_MANIFEST_PATH, PROMOTION_MANIFEST_PATH)
    if not PROMOTION_ISSUER:
        raise ValueError('set a nonempty accountable PROMOTION_ISSUER')
    review_blinding_key = _read_protected_key(REVIEW_KEY_ENV)
    threshold_source_replay_inputs = None
    try:
        calibration_candidate = load_candidate_evaluation_manifest(
            CALIBRATION_CANDIDATE_MANIFEST_PATH, model_bundle=preliminary_bundle,
            marker_registry=marker_registry, typing_registry=typing_registry,
            source_run_manifest=source_run_manifest, allow_locked_test=False,
        )
        threshold_source_replay_inputs = ThresholdSelectionReplayInputs(
            preliminary_bundle=preliminary_bundle,
            calibration_candidate_manifest=calibration_candidate,
            marker_registry=marker_registry, typing_registry=typing_registry,
            source_run_manifest=source_run_manifest,
            calibration_sample_frames=_load_calibration_frames(CALIBRATION_REVIEW_ARTIFACTS, 'sample'),
            calibration_reviewer_frames=_load_calibration_frames(CALIBRATION_REVIEW_ARTIFACTS, 'reviewer'),
            calibration_review_ledgers=_load_calibration_frames(CALIBRATION_REVIEW_ARTIFACTS, 'ledger'),
            review_blinding_key=review_blinding_key,
        )
        release_signing_key = _read_protected_key(RELEASE_KEY_ENV)
        try:
            promotion_manifest = TypingModelPromotionManifest.create(
                promotion_version=PROMOTION_VERSION, model_bundle=frozen_bundle,
                policy=promotion_policy, validation_report=validation_report,
                locked_test_report=locked_test_report,
                threshold_source_replay_inputs=threshold_source_replay_inputs,
                marker_registry=marker_registry, typing_registry=typing_registry,
                review_blinding_key=review_blinding_key,
                release_signing_key=release_signing_key, issuer=PROMOTION_ISSUER,
                issued_at_utc=datetime.now(timezone.utc).isoformat(),
            )
            promotion_manifest.write_json(PROMOTION_MANIFEST_PATH)
        finally:
            del release_signing_key
    finally:
        del review_blinding_key
        if threshold_source_replay_inputs is not None:
            del threshold_source_replay_inputs


## Release disposition

Archive the validation report before the locked opening, then archive the locked report and promotion manifest. Record the operator disposition in the study's governed audit/decision ledger; this template does not invent a separate ledger schema. If either exact policy check or any source replay fails, preserve the artifacts and report the failure. Do not change thresholds, class rules, samples, exclusions, or upstream processing using held-out results. A corrected candidate starts a new development/validation cycle with a genuinely future locked set. Only a separately configured frozen model bundle plus its verified promotion manifest may enter production; reviewed cells are evidence, never production overrides.
